# DEA Wetlands Insight Tool: multi-scene processing with STAC

The Wetlands Insight Tool (WIT) summarises changes in open water, wetness, green vegetation, dry vegetation and bare soil. This notebook follows the [DEA WIT notebook](https://github.com/GeoscienceAustralia/dea-notebooks/blob/stable/DEA_products/DEA_Wetlands_Insight_Tool.ipynb), but processes large wetlands spanning multiple Landsat path/rows using the time aggregation approach from [wit_tooling](https://github.com/GeoscienceAustralia/wit_tooling). No Datacube index is required: observations are discovered using DEA's STAC API and read from public DEA storage.

Each feature in the input shapefile is analysed separately. The output is a normalised WIT graph and CSV for each feature. The calculation uses Landsat Collection 3 products; it should not be expected to reproduce an older Collection 2 WIT product exactly.

**Learning goals:** discover matching satellite products, inspect the time windows,
understand how scenes are combined across path/rows, and interpret the quality
and normalised fractions in a WIT graph. Run the cells in order.

## Background

Wetlands support biodiversity, water quality and carbon storage. WIT provides a consistent history of surface water and vegetation changes to help investigate wetland responses to floods, drought and management. Open water, wetness and dry land are mutually exclusive at each valid pixel; green vegetation, dry vegetation and bare soil describe fractions of the dry-land component. Interpret changes alongside field knowledge and the quality information in the CSV.

### Expected output

A stacked plot with open water, wetness, green vegetation, dry vegetation and bare soil, in the same style as the [published WIT graph](https://dea-public-data.s3-ap-southeast-2.amazonaws.com/derivative/ga_ls_wit_ramsar_class_myear_3/1-0-0/1986--P35Y/271.png). Vegetation is normalised for display so the five proportions sum to one wherever the vegetation total is nonzero. A CSV retains both original and normalised proportions, the missing-pixel fraction and the number of scenes in each composite.

## WIT calculation and multi-scene aggregation

For each polygon, Landsat surface reflectance, fractional cover (FC) and Water Observations from Space (WOfS) items are matched by satellite, path/row and acquisition time. In the spirit of `wit_tooling`'s `--aggregate 15` mode, the first available scene starts a 15-day window. Scenes with acquisition times strictly before the end of that window fill pixels that were not valid in earlier scenes; the first subsequent scene starts the next window. The composite is summarised only after scenes have been combined, so adjacent path/rows and observations on different days can contribute to one result.

WOfS identifies open water and excludes cloud, cloud shadow and non-contiguous pixels. On non-water pixels, Landsat fmask and valid reflectance are required; Tasseled Cap Wetness (TCW) at or above −350 is wet, and other pixels contribute fractional cover. Earlier valid open water or land is not overwritten by later observations. Like WIT tooling, the polygon raster includes every touched pixel and a plotted composite must have **less than 10% missing**. Unlike the original code's per-band aggregation, the first valid WIT class is kept together at each pixel so that a later dry observation cannot also be counted over earlier water.

To bound memory, the polygon is rasterised and processed in tiles; only the per-window summaries are retained. Completed windows are checkpointed and reused on rerun. A large polygon and a multi-year range can require extensive network transfer and hours or longer; start with a short range first.

For example, if the first available acquisition is on 1 January, the first
window contains observations strictly before 16 January. The next available
acquisition starts a new window; it need not be exactly on 16 January. Scene
order matters *within* a window: once a pixel has a valid class, a later
observation cannot replace it. Different parts of the polygon may therefore
come from different scenes or dates.

## Getting Started

### Load packages

This notebook contains its own WIT processing and plotting functions. It does
not require `dea-conflux`, a Datacube index, or the repository containing this
notebook. It does require the following Python packages and network access to
DEA STAC and public S3 data. Install them into the environment used by your
Jupyter kernel, for example:

```bash
python -m pip install geopandas numpy pandas rasterio shapely matplotlib seaborn \
    "odc-stac[botocore]" pystac-client ipykernel jupyterlab
```

With uv, use `uv pip install --python /path/to/venv/bin/python` followed by the
same package names. `botocore` configures unsigned access to public S3 data.
Select that environment as the notebook kernel and run the cells in order. The
following definition cells explain the steps and can be inspected or modified.

In [ ]:
import datetime as dt
import hashlib
import json
import logging
import os
from dataclasses import dataclass
from pathlib import Path

import geopandas as gpd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import odc.stac
import pandas as pd
import pystac_client
import rasterio.features
import seaborn as sns
from IPython.display import display
from odc.geo import resyx_
from odc.geo.geobox import GeoBox
from odc.geo.geom import Geometry
from shapely.geometry import box, shape

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("wit_multiscene")

### Define scene and time-window settings

An ARD, fractional-cover and WOfS item from the same acquisition form one
`Scene`. The window function starts at the first scene and keeps later scenes
strictly before the window's end. These definitions are used both for the
preview and the calculation. The algorithm version is part of every checkpoint
key: change it if you change the pixel calculation or checkpoint format.

In [ ]:
# Collection 3 products and output components used by the calculation.
ARD_PRODUCTS = ("ga_ls5t_ard_3", "ga_ls7e_ard_3", "ga_ls8c_ard_3", "ga_ls9c_ard_3")
FC_PRODUCT = "ga_ls_fc_3"
WO_PRODUCT = "ga_ls_wo_3"
REFLECTANCE = tuple(
    f"nbart_{band}" for band in ("blue", "green", "red", "nir", "swir_1", "swir_2")
)
RESULT_BANDS = ("water", "wet", "pv", "npv", "bs")

# Change this version if the calculation or checkpoint schema changes.
ALGORITHM_VERSION = "wit-stac-1"


@dataclass(frozen=True)
class Scene:
    time: dt.datetime
    ard: object
    fc: object
    water: object


def time_windows(items, days):
    """Partition ordered observations into non-overlapping, start-anchored windows."""
    if days < 1:
        raise ValueError("Composite window must be at least one day")
    index = 0
    while index < len(items):
        end = index + 1
        limit = items[index][0] + dt.timedelta(days=days)
        while end < len(items) and items[end][0] < limit:
            end += 1
        yield items[index:end]
        index = end

### Connect to DEA's STAC API

Configure unsigned access to public S3 assets, then open the DEA Explorer STAC catalogue. Unlike the Datacube-index based Conflux command, this notebook does not need a database connection.

In [ ]:
odc.stac.configure_s3_access(cloud_defaults=True, aws_unsigned=True)
catalog = pystac_client.Client.open("https://explorer.dea.ga.gov.au/stac")

### Analysis parameters

Set `shapefile` to a polygon or multipolygon dataset, `id_field` to a unique, non-null feature identifier, and choose a date range. The start is inclusive and the end is **exclusive**: the default covers the two years from 1 January 2022 up to 1 January 2024. The example uses `fid` from the supplied Ramsar shapefile. Set `id_field = None` to use the file's row index instead, or set `name_field` to a column to use as the plot title.

For a historical analysis, try `start_date = "1986-01-01"` and an end date after the latest desired day. Availability depends on the catalogue and satellite record. The Landsat 5 and 7 inclusion periods follow the historical predicates in `wit_tooling` (including Landsat 7 SLC-off observations during 2010–2013); Landsat 8 and 9 are included when available. They are not identical to the single-scene notebook's Landsat 7 filter. Test a shorter range before attempting the full history.

The `window_days` parameter controls the compositing interval, not the
frequency of satellite acquisitions. `tile_size` controls memory use, not the
30 m output resolution. Smaller tiles use less memory but can increase
overhead. Keep the output directory to reuse completed windows after a restart.

In [ ]:
shapefile = Path.home() / "ramsarbig/ramsarbig.shp"
id_field = "fid"       # Set to None to use the row index.
name_field = None      # Optional column used in graph titles.

start_date = "2022-01-01"
end_date = "2024-01-01"  # Exclusive: includes all of 2022 and 2023.
window_days = 15

resolution = 30         # Metres in Australian Albers (EPSG:3577).
tile_size = 1024        # Pixels per side; reduce if memory is limited.
output_dir = Path("wit_multiscene_output")

### Load a shapefile that describes your wetland study area

Use 2D polygon geometries with no Z coordinates, as in the original notebook. Geometries are reprojected to Australian Albers (EPSG:3577). Unlike the original notebook's `poly.geometry[0]`, every feature is processed independently, including overlapping polygons. Identifiers must be unique and safe to use as filenames.

In [ ]:
# Check the geometry before changing its coordinate reference system.
poly = gpd.read_file(shapefile)
if poly.crs is None:
    raise ValueError("The shapefile must have a coordinate reference system")
if poly.empty or not poly.geometry.geom_type.isin(["Polygon", "MultiPolygon"]).all():
    raise ValueError("Provide at least one polygon or multipolygon")
if poly.geometry.is_empty.any() or not poly.geometry.is_valid.all():
    raise ValueError("All polygon geometries must be non-empty and valid")
if poly.geometry.has_z.any():
    raise ValueError("Remove Z coordinates before processing")

# One uniquely identified feature produces one graph and one CSV.
poly = poly.to_crs("EPSG:3577")
if id_field is not None:
    poly = poly.set_index(id_field)
if poly.index.hasnans or not poly.index.is_unique or len(set(map(str, poly.index))) != len(poly):
    raise ValueError("Feature IDs must be unique and non-null")
for feature_id in poly.index:
    name = str(feature_id)
    if name in ("", ".", "..") or "/" in name or "\\" in name:
        raise ValueError(f"Feature ID cannot be used as a filename: {name!r}")
if name_field is not None and name_field not in poly.columns:
    raise ValueError(f"Missing name field: {name_field}")
output_dir.mkdir(parents=True, exist_ok=True)
display(poly)

## Load Landsat, Water Observations and Fractional Cover

### Search for matching observations

Each polygon is searched independently using its WGS84 bounds and footprint. The STAC search covers Landsat 5, 7, 8 and 9 ARD, WOfS and FC. An ARD item without both matching FC and WOfS is logged and skipped; conflicting duplicate items cause an error rather than mixing scenes. Search results are sorted by acquisition time before making windows.

### Define the STAC scene search

The following functions match items using satellite, path/row and acquisition
time, apply the historical satellite selection, and return scenes in time order.
The bounding-box search is refined to the polygon footprint. Items missing a
matching product are counted and logged; duplicate matches raise an error.

In [ ]:
def _scene_key(item):
    properties = item.properties
    platform = properties.get("platform")
    region = properties.get("odc:region_code")
    if not platform or not region or item.datetime is None:
        raise ValueError(f"STAC item {item.id} lacks platform, region or acquisition time")
    return platform, region, item.datetime


def _allowed_satellite(item):
    """Apply wit_tooling's historical Landsat 5/7 selection; include Landsat 9."""
    date = item.datetime.date()
    platform = item.properties["platform"]
    if platform == "landsat-5":
        return date <= dt.date(1999, 12, 31) or dt.date(2003, 1, 1) <= date <= dt.date(2011, 12, 31)
    if platform == "landsat-7":
        return date <= dt.date(2003, 5, 31) or dt.date(2010, 1, 1) <= date <= dt.date(2013, 5, 31)
    if platform == "landsat-8":
        return date >= dt.date(2013, 1, 1)
    return platform == "landsat-9"


def search_scenes(catalog, polygon, start, end):
    """Find complete ARD/FC/WOfS scene triplets. End date is exclusive."""
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    if start.tzinfo is not None or end.tzinfo is not None:
        raise ValueError("Use timezone-naive calendar dates")
    if start >= end:
        raise ValueError("The exclusive end date must be after the start date")
    footprint = polygon.to_crs("EPSG:4326")
    bbox = list(footprint.boundingbox)
    interval = f"{start.isoformat()}Z/{(end - pd.Timedelta(microseconds=1)).isoformat()}Z"
    products = [*ARD_PRODUCTS, FC_PRODUCT, WO_PRODUCT]
    grouped = {name: {} for name in products}
    for item in catalog.search(
        collections=products, bbox=bbox, datetime=interval, method="GET", limit=500
    ).items():
        product = item.collection_id
        if product not in grouped:
            raise ValueError(f"Unexpected STAC collection {product}")
        if not shape(item.geometry).intersects(footprint.geom):
            continue
        key = _scene_key(item)
        if not _allowed_satellite(item):
            continue
        if key in grouped[product]:
            raise ValueError(f"Multiple {product} STAC items for scene {key}")
        grouped[product][key] = item

    scenes = []
    skipped = 0
    for product in ARD_PRODUCTS:
        for key, ard in grouped[product].items():
            if key not in grouped[FC_PRODUCT] or key not in grouped[WO_PRODUCT]:
                skipped += 1
                continue
            scenes.append(Scene(ard.datetime, ard, grouped[FC_PRODUCT][key], grouped[WO_PRODUCT][key]))
    if skipped:
        logger.warning("Skipped %d ARD scenes without matching FC and WOfS", skipped)
    scenes.sort(key=lambda scene: (scene.time, scene.ard.id))
    return scenes

In [ ]:
scene_groups = {}
for feature_id, feature in poly.iterrows():
    polygon = Geometry(feature.geometry, str(poly.crs))
    scenes = search_scenes(catalog, polygon, start_date, end_date)
    if not scenes:
        raise ValueError(f"No complete observations for polygon {feature_id}")
    scene_groups[feature_id] = scenes
    print(f"{feature_id}: {len(scenes)} matched Landsat/FC/WOfS scenes")

### Preview the composite windows

Before loading raster pixels, inspect the first few windows. The number of
scenes is not the number of cloud-free pixels: a scene can cover only part of
the polygon or contain unusable observations.

In [ ]:
window_rows = []
for feature_id, scenes in scene_groups.items():
    observations = [(scene.time, scene) for scene in scenes]
    for window in time_windows(observations, window_days):
        window_rows.append({
            "feature_id": feature_id,
            "first_acquisition_utc": window[0][0],
            "last_acquisition_utc": window[-1][0],
            "scene_count": len(window),
        })
display(pd.DataFrame(window_rows).head(10))

### Load and mask data; calculate Tasseled Cap Wetness

For each tile and scene, the following functions load WOfS using nearest-neighbour resampling, reflectance using bilinear resampling, and fmask/FC using nearest-neighbour resampling. It computes the Crist (1985) TCW index from the six NBART bands, applies the −350 threshold and masks cloud, shadow, non-contiguous and nodata pixels. FC values are divided by 100. Open-water pixels need valid WOfS; dry-land and wet pixels additionally need valid reflectance and fmask.

### Composite and summarise

`run_windows` builds first-valid composites in chronological order, sums each WIT component across the rasterised polygon, and divides by the number of valid pixels. `pc_missing` is relative to all polygon pixels. This is equivalent to processing the common full grid but avoids loading the entire grid in memory. The checkpoint key includes the geometry, resolution, algorithm version and input scene IDs. Re-running the same analysis reuses completed windows, including after an interruption; changing the input data or geometry recomputes affected windows.

For a dry pixel, FC contributes three continuous fractions rather than a single
dominant class. The composite sums these fractions alongside counts of water and
wet pixels and divides by the number of valid polygon pixels. Missing pixels
are reported separately; they are not treated as dry land.

### Define how a scene is classified

The following functions load just one tile at a time. WOfS supplies open water
and its quality mask. For valid non-water pixels, reflectance and fmask determine
wetness; fractional cover is loaded only where dry-land pixels remain. `available`
ensures that pixels already filled by an earlier scene cannot be overwritten.

In [ ]:
def _load(item, bands, tile, resampling):
    data = odc.stac.load(
        [item], bands=bands, geobox=tile, resampling=resampling,
        fail_on_error=True,
    )
    if data.sizes.get("time") != 1:
        raise ValueError(f"Expected one time step for STAC item {item.id}")
    return data.isel(time=0, drop=True)


def _not_nodata(array):
    values = array.values
    valid = np.isfinite(values)
    nodata = array.attrs.get("nodata")
    if nodata is not None and not np.isnan(nodata):
        valid &= values != nodata
    return valid


def _scene_pixels(scene, tile, available):
    water = _load(scene.water, ["water"], tile, "nearest")["water"]
    water_values = water.values
    quality = _not_nodata(water) & ((water_values & 0b01100011) == 0) & available
    open_water = quality & ((water_values & 128) != 0)
    land_candidate = quality & ~open_water
    if not land_candidate.any():
        return open_water, np.zeros(available.shape, bool), None, None

    ard = _load(scene.ard, [*REFLECTANCE, "oa_fmask"], tile, {
        "oa_fmask": "nearest", "*": "bilinear",
    })
    valid_land = land_candidate & np.isin(ard.oa_fmask.values, (1, 5))
    for band in REFLECTANCE:
        valid_land &= _not_nodata(ard[band])
    if not valid_land.any():
        return open_water, valid_land, None, None

    tcw = (
        0.0315 * ard.nbart_blue.values + 0.2021 * ard.nbart_green.values
        + 0.3102 * ard.nbart_red.values + 0.1594 * ard.nbart_nir.values
        - 0.6806 * ard.nbart_swir_1.values - 0.6109 * ard.nbart_swir_2.values
    )
    wet = valid_land & (tcw >= -350)
    dry = valid_land & ~wet
    fc = _load(scene.fc, ["pv", "npv", "bs"], tile, "nearest") if dry.any() else None
    return open_water, wet, dry, fc

### Define the tile composite

The next function rasterises the polygon on a 30 m grid and walks over its
tiles. For each tile, intersecting scenes are considered in time order. It sums
water, wetness and fractional-cover contributions, then divides by the valid
pixel count. The missing fraction uses the total number of polygon pixels.

In [ ]:
def composite_window(scenes, geometry, crs="EPSG:3577", resolution=30, tile_size=1024):
    """Return polygon means for a window, using the first valid class per pixel."""
    if not scenes or resolution <= 0 or tile_size <= 0:
        raise ValueError("A non-empty window, positive resolution and tile size are required")
    polygon = Geometry(geometry, crs)
    grid = GeoBox.from_geopolygon(polygon, resolution=resyx_(-resolution, resolution))
    footprints = [
        Geometry(shape(scene.ard.geometry), "EPSG:4326").to_crs(crs).geom
        for scene in scenes
    ]
    totals = dict.fromkeys(RESULT_BANDS, 0.0)
    total = valid_count = 0
    processed_tiles = 0
    for y in range(0, grid.shape.y, tile_size):
        for x in range(0, grid.shape.x, tile_size):
            tile = grid[y:y + tile_size, x:x + tile_size]
            inside = rasterio.features.rasterize(
                [(geometry, 1)], out_shape=tile.shape, transform=tile.transform,
                all_touched=True, dtype="uint8",
            ).astype(bool)
            if not inside.any():
                continue
            processed_tiles += 1
            if processed_tiles % 20 == 0:
                logger.info("Compositing tile %d", processed_tiles)
            total += int(inside.sum())
            filled = np.zeros(inside.shape, dtype=bool)
            tile_extent = box(*tile.extent.boundingbox)
            for scene, footprint in zip(scenes, footprints):
                if not footprint.intersects(tile_extent):
                    continue
                open_water, wet, dry, fc = _scene_pixels(scene, tile, inside & ~filled)
                totals["water"] += int(open_water.sum())
                totals["wet"] += int(wet.sum())
                if dry is not None and dry.any():
                    for band in ("pv", "npv", "bs"):
                        values = fc[band].values
                        valid_fc = dry & _not_nodata(fc[band])
                        totals[band] += float(values[valid_fc].sum(dtype="float64")) / 100
                    filled |= dry
                filled |= open_water | wet
                if np.all(filled | ~inside):
                    break
            valid_count += int(filled.sum())
    if total == 0:
        raise ValueError("Polygon contains no raster pixels")
    result = {
        band: totals[band] / valid_count if valid_count else float("nan")
        for band in RESULT_BANDS
    }
    result["pc_missing"] = 1 - valid_count / total
    return result

### Define the resumable window calculation

Each window's checkpoint is keyed by polygon, CRS, resolution, algorithm version
and scene IDs. Completed summaries are written atomically and reused on rerun.
Window boundaries depend on the first available acquisition, so overlapping date
ranges only share a checkpoint when their window scenes match exactly.

In [ ]:
def _checkpoint_path(cache_dir, geometry, crs, resolution, scenes):
    payload = {
        "algorithm": ALGORITHM_VERSION,
        "geometry": geometry.wkb_hex,
        "crs": str(crs),
        "resolution": resolution,
        "scenes": [[scene.ard.id, scene.fc.id, scene.water.id] for scene in scenes],
    }
    digest = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    return Path(cache_dir) / f"{digest}.json"


def run_windows(scenes, geometry, cache_dir, crs="EPSG:3577", resolution=30,
                tile_size=1024, window_days=15):
    """Compute or resume one independently checkpointed result per time window."""
    if not scenes:
        raise ValueError("No matching Landsat, FC and WOfS scenes found")
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    items = [(scene.time, scene) for scene in scenes]
    rows = []
    for number, window in enumerate(time_windows(items, window_days), start=1):
        window_scenes = [scene for _, scene in window]
        path = _checkpoint_path(cache_dir, geometry, crs, resolution, window_scenes)
        if path.exists():
            with path.open() as stream:
                row = json.load(stream)
            status = "cached"
        else:
            print(f"Computing window {number}: {window[0][0].date()}, "
                  f"{len(window)} scenes", flush=True)
            row = composite_window(window_scenes, geometry, crs, resolution, tile_size)
            row["date"] = window[0][0].isoformat()
            row["scene_count"] = len(window)
            tmp = path.with_suffix(f".{os.getpid()}.tmp")
            try:
                with tmp.open("w") as stream:
                    json.dump(row, stream)
                os.replace(tmp, path)
            finally:
                if tmp.exists():
                    tmp.unlink()
            status = "computed"
        rows.append(row)
        print(f"Window {number}: {row['date'][:10]}, {len(window)} scenes, "
              f"{row['pc_missing']:.1%} missing ({status})", flush=True)
    result = pd.DataFrame(rows)
    result["date"] = pd.to_datetime(result["date"], utc=True).dt.tz_localize(None)
    return result

In [ ]:
polygon_results = {}
for feature_id, feature in poly.iterrows():
    print(f"Processing polygon {feature_id}")
    polygon_results[feature_id] = run_windows(
        scene_groups[feature_id], feature.geometry,
        cache_dir=output_dir / "checkpoints",
        crs=str(poly.crs), resolution=resolution,
        tile_size=tile_size, window_days=window_days,
    )

## Normalise Fractional Cover Values in WIT result

As in the original notebook, normalisation does not change open water or wetness. It rescales the three FC values to the remaining vegetation area, `1 − water − wet`. Both unnormalised and normalised values are retained. If the FC total is zero while vegetation area remains, normalisation is undefined and the normalised values are reported as missing rather than inventing a composition.

For example, if water is 0.2 and wet is 0.1, the three normalised FC fractions
must share the remaining 0.7. Each is multiplied by
`0.7 / (pv + npv + bs)`. This compensates for small deviations of FC sums from
one; it does not fill missing pixels or improve the underlying observations.

### Define fractional-cover normalisation

The function below preserves water and wetness and scales the three dry-land
fractions to the remaining area. It reports undefined normalisation rather than
inventing a composition when that area is nonzero but the FC sum is zero.

In [ ]:
def normalise_results(frame):
    """Normalise FC means to the portion not classified as water or wet."""
    result = frame.copy()
    vegetation_area = 1 - result.water - result.wet
    vegetation_sum = result.pv + result.npv + result.bs
    for band in ("pv", "npv", "bs"):
        result[f"norm_{band}"] = result[band]
        mask = vegetation_sum > 0
        result.loc[mask, f"norm_{band}"] = (
            result.loc[mask, band] / vegetation_sum[mask] * vegetation_area[mask]
        )
    undefined = (vegetation_sum == 0) & (vegetation_area > 0)
    if undefined.any():
        logger.warning("Cannot normalise %d observations with zero vegetation total", undefined.sum())
        result.loc[undefined, ["norm_pv", "norm_npv", "norm_bs"]] = np.nan
    return result

In [ ]:
polygon_results = {
    feature_id: normalise_results(frame)
    for feature_id, frame in polygon_results.items()
}
for feature_id, frame in polygon_results.items():
    usable = frame["pc_missing"] < 0.1
    print(f"Polygon {feature_id}: {usable.sum()} of {len(frame)} windows meet the graph quality rule")
    display(frame.head())

## Create WIT Plot

Each plot is displayed and saved as `<feature_id>.png` in `output_dir`. As in WIT tooling, observations with 10% or more missing pixels are excluded from the graph; they remain in the CSV for quality assessment. If no observation passes this threshold, the saved graph is labelled accordingly. The single-observation case is shown over a 24-hour span to make the stacked areas visible. Shorter-than-one-year plots use monthly labels; longer plots use yearly labels.

Read the graph vertically: at each date the five coloured areas add to 100%
when normalisation is defined. The graph omits low-quality observations and
interpolates between the remaining dates; it does not show daily measurements.
Use the CSV to inspect the actual acquisition dates and missing fractions.

Hatching highlights the Landsat 5/8 gap and sparse qualifying observations, not a measured proportion.

### Define the graph

The plotting functions retain the WIT colours and draw the five normalised
components as a percentage of valid area. They exclude windows with at least
10% missing pixels. Hatched regions mark the Landsat 5/8 gap and periods of
sparse qualifying observations; gaps can otherwise be visually interpolated.
The CSV remains the source for individual dates and missing-pixel fractions.

In [ ]:
def generate_low_quality_data_periods(df):
    # default: all data points are good
    df.loc[:, "off_value"] = 0

    # Add the first no-data times (SLC-off only)
    ls5_8_gap_start = dt.datetime(2011, 11, 1)
    ls5_8_gap_end = dt.datetime(2013, 4, 1)

    df.loc[
        (df["date"] >= ls5_8_gap_start) & (df["date"] <= ls5_8_gap_end),
        "off_value",
    ] = 100

    # periods with an observation density of less than four observations within a twelve month (365 days) period
    for i in range(3, len(df) - 3):
        # can change to another threshold (like: 100 days) to test dynamic no-data-period display
        start_date = df.iloc[i]["date"]
        end_date = df.iloc[i + 3]["date"]
        if (end_date - start_date).days > 365:
            df.loc[
                (df["date"] >= start_date) & (df["date"] <= end_date),
                "off_value",
            ] = 100

    return df


def display_wit_stack_with_df(
    polygon_base_df,
    polygon_name="your_wetland_name",
    png_name="your_file_name",
    width=32,
    height=6,
    x_axis_labels="years",
):
    """
    This functions produces WIT plots. Function displays a stack plot and saves as a png.

    Last modified: July 2023

    Parameters
    ----------
    polygon_base_df : pandas DataFrame with columns including:
    ['date',
     'wet',
     'water',
     'norm_bs',
     'norm_pv',
     'norm_npv']
     polygon_name : string
     png_name : string
     x_axis_labels : string with options of "years" or "months"
     to set either years or months on the x axis as labels


    """

    plt.rcParams["axes.facecolor"] = "white"
    plt.rcParams["savefig.facecolor"] = "white"
    plt.rcParams["text.usetex"] = False

    fig = plt.figure()
    fig.set_size_inches(width, height)
    ax = fig.add_subplot(111)
    ax.autoscale(enable=True)

    pal = [
        sns.xkcd_rgb["cobalt blue"],
        sns.xkcd_rgb["neon blue"],
        sns.xkcd_rgb["grass"],
        sns.xkcd_rgb["beige"],
        sns.xkcd_rgb["brown"],
    ]

    plt.title(
        f"Percentage of area dominated by WOfS, Wetness, Fractional Cover for\n {polygon_name}",
        fontsize=16,
    )

    ax.stackplot(
        polygon_base_df["date"],
        polygon_base_df["water"] * 100,
        polygon_base_df["wet"] * 100,
        polygon_base_df["norm_pv"] * 100,
        polygon_base_df["norm_npv"] * 100,
        polygon_base_df["norm_bs"] * 100,
        colors=pal,
        alpha=0.7,
    )

    # manually change the legend display order
    legend = ax.legend(
        ["open water", "wet", "green veg", "dry veg", "bare soil"][::-1],
        loc="lower left",
    )
    handles = legend.legend_handles

    for i, handle in enumerate(handles):
        handle.set_facecolor(pal[::-1][i])
        handle.set_alpha(0.7)

    # setup the display ranges
    ax.set_ylim(0, 100)
    ax.set_xlim(polygon_base_df["date"].min(), polygon_base_df["date"].max())

    # add a new column: 'off_value' based on low quality data setting
    polygon_base_df = generate_low_quality_data_periods(polygon_base_df)

    ax.fill_between(
        polygon_base_df["date"],
        0,
        100,
        where=polygon_base_df["off_value"] == 100,
        color="white",
        alpha=0.5,
        hatch="//",
    )

    if x_axis_labels == "years":
        # modify the xaxis settings
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    if x_axis_labels == "months":
        # modify the xaxis settings
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%Y"))
        # Rotates and right-aligns the x labels so they don't crowd each other.
        for label in ax.get_xticklabels(which="major"):
            label.set(rotation=30, horizontalalignment="right")

    x_label_text = (
        "The Fractional Cover algorithm developed by the Joint Remote Sensing Research Program and\n"
        " the Water Observations from Space algorithm developed by Geoscience Australia are used in "
        "the production of this data"
    )

    ax.set_xlabel(x_label_text, style="italic")

    plt.savefig(f"{png_name}.png", bbox_inches="tight")
    plt.show()

    plt.close(fig)


def plot_results(frame, polygon_name, png_name):
    """Display and save a normalised WIT graph, excluding >=10% missing."""
    graph = frame.loc[frame.pc_missing < 0.1].copy()
    if graph.empty:
        logger.warning("No observations with less than 10%% missing for %s", polygon_name)
        graph = frame.copy()
        graph.loc[:, ["water", "wet", "norm_pv", "norm_npv", "norm_bs"]] = np.nan
        polygon_name = f"{polygon_name} (no observations with less than 10% missing)"
    elif len(graph) == 1:
        polygon_name = f"{polygon_name} (single observation)"
    if len(graph) == 1:
        graph = pd.concat([graph, graph], ignore_index=True)
        graph.loc[0, "date"] -= pd.Timedelta(hours=12)
        graph.loc[1, "date"] += pd.Timedelta(hours=12)
    labels = "months" if graph.date.max() - graph.date.min() < pd.Timedelta(days=365) else "years"
    display_wit_stack_with_df(graph, polygon_name, str(png_name), x_axis_labels=labels)

In [ ]:
for feature_id, frame in polygon_results.items():
    polygon_name = str(poly.loc[feature_id, name_field]) if name_field else f"Wetland {feature_id}"
    plot_results(frame, polygon_name, output_dir / str(feature_id))

## Create WIT Comma-separated values (CSV) output file

The CSV contains one row per composite window. Dates are the UTC acquisition times of the first scene in each window; `scene_count` is the number of matched scenes in that window. Fractions use the range 0–1 where applicable. `pc_missing` includes clouds, unobserved pixels and pixels not covered by a valid scene. `norm_pv`, `norm_npv` and `norm_bs` are the normalised green, dry and bare-soil components.

In [ ]:
for feature_id, frame in polygon_results.items():
    csv_path = output_dir / f"{feature_id}.csv"
    frame.to_csv(csv_path, index=False)
    print(f"Wrote {csv_path} and {output_dir / (str(feature_id) + '.png')}")

## Troubleshooting and interpretation

- **No complete scenes:** check the dates, polygon and catalogue connectivity.
  An incomplete ARD/FC/WOfS triplet is reported and skipped.
- **Empty graph:** inspect `pc_missing` in the CSV. Windows with 10% or more
  missing pixels are deliberately excluded; a labelled empty graph is not a
  zero-water result.
- **Slow or interrupted run:** retry with the same output directory to reuse
  completed checkpoints. Try a short date range first, and reduce `tile_size`
  if memory is limited. The full historical run can take substantially longer.
- **Comparison with published WIT:** this notebook uses Collection 3, historical
  satellite filters and a first-valid-class composite. Differences from older
  Collection 2 or per-band aggregation products are expected.

## Additional information

- [WIT tooling and aggregation](https://github.com/GeoscienceAustralia/wit_tooling)
- [DEA Wetlands Insight Tool information](https://knowledge.dea.ga.gov.au/data/product/dea-wetlands-insight-tool/)
- Dunn et al. (2023), [Wetlands Insight Tool](https://doi.org/10.1007/s13157-023-01682-7)
- Crist (1985), [Tasseled Cap transformation](https://doi.org/10.1016/0034-4257(85)90102-6)

Notebook code is licensed under Apache 2.0. DEA data are licensed under Creative Commons Attribution 4.0.